# Resultados do benchmark e explicabilidade

**Material de apoio em português brasileiro.** Este notebook usa exclusivamente `dados.json`, cujos valores foram transcritos do estudo anterior e conferidos contra a edição de seis páginas. Não carrega pesos, não acessa imagens e não executa novos experimentos. ROC-AUC não é porcentagem de acerto.

Os arquivos `apresentacao.pptx`, `apresentacao.pdf` e `apresentacao.html` contêm o mesmo roteiro visual. As notas completas estão em `roteiro.md`.

In [1]:
from pathlib import Path
from decimal import Decimal
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

folder = Path.cwd() if (Path.cwd() / 'dados.json').is_file() else Path.cwd() / 'presentation'
assert (folder / 'dados.json').is_file(), 'Abra o notebook na pasta presentation ou na raiz do repositório.'
data = json.loads((folder / 'dados.json').read_text(encoding='utf-8'))
print(data['proveniencia'])

Médias arredondadas do estudo anterior, reproduzidas na seção V-A da edição de seis páginas. Não são uma reavaliação dos pesos fine-tuned atuais. As diferenças são subtrações das médias reportadas; não são testes de significância.


## 1. Teste limpo versus degradado

As médias são do benchmark anterior: cinco famílias treinadas do zero e a configuração DINOv3 congelada. Elas não caracterizam os checkpoints fine-tuned da extensão. A queda é a diferença aritmética das médias arredondadas. Não é uma estimativa de significância.

In [2]:
rgb = pd.DataFrame(data['rgb'])
rgb['queda_auc'] = [float(Decimal(str(a)) - Decimal(str(b))) for a, b in zip(rgb.limpo, rgb.degradado)]
display(rgb.round(3))
assert len(rgb) == 6
assert rgb.loc[rgb.modelo.eq('Xception'), 'queda_auc'].iloc[0] == 0.275
assert rgb.loc[rgb.modelo.eq('DINOv3'), 'queda_auc'].iloc[0] == 0.083

,modelo,regime,limpo,degradado,queda_auc
0,Xception,do zero,0.884,0.609,0.275
1,ResNet-18,do zero,0.846,0.635,0.211
2,MobileNetV3,do zero,0.839,0.594,0.245
3,ViT,do zero,0.624,0.608,0.016
4,CLIP-style,do zero; sem pesos CLIP,0.750,0.619,0.131
5,DINOv3,configuração congelada do estudo anterior,0.809,0.726,0.083


In [3]:
ordered = rgb.sort_values('limpo', ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5.8))
for i, row in ordered.iterrows():
    ax.plot([row.limpo, row.degradado], [i, i], linewidth=1, alpha=0.4)
ax.scatter(ordered.limpo, range(len(ordered)), label='Teste limpo', marker='o', s=65)
ax.scatter(ordered.degradado, range(len(ordered)), label='Teste degradado', marker='s', s=65)
ax.set_yticks(range(len(ordered)), ordered.modelo)
ax.invert_yaxis()
ax.set_xlim(0.5, 1.0)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'.replace('.', ',')))
ax.set_xlabel('ROC-AUC — médias reportadas; sem intervalos inferidos')
ax.set_title('A degradação muda o ranking')
ax.legend(frameon=False)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)

**Leitura:** Xception lidera o teste limpo (0,884); DINOv3 tem a maior média no degradado (0,726) entre estas seis configurações. A diferença de regime de pré-treinamento impede atribuir essa comparação apenas à arquitetura. A pequena queda do ViT (0,016) ocorre a partir de uma média limpa já baixa (0,624).

## 2. RGB versus entradas híbridas no teste degradado

Os três exemplos são contrastes descritivos reportados. Não representam todos os modos possíveis e não demonstram causalidade nem generalização para outros datasets.

In [4]:
hybrid = pd.DataFrame(data['hibridos'])
hybrid['ganho_auc'] = [float(Decimal(str(a)) - Decimal(str(b))) for a, b in zip(hybrid.hibrido_degradado, hybrid.rgb_degradado)]
display(hybrid.round(3))
assert hybrid.ganho_auc.tolist() == [0.041, 0.013, 0.018]

,modelo,representacao,rgb_degradado,hibrido_degradado,ganho_auc
0,Xception,RGB + magnitude,0.609,0.650,0.041
1,ResNet-18,RGB + pilha de frequência,0.635,0.648,0.013
2,MobileNetV3,RGB + magnitude,0.594,0.612,0.018


In [5]:
fig, ax = plt.subplots(figsize=(10, 4.5))
positions = list(range(len(hybrid)))
ax.scatter(hybrid.rgb_degradado, positions, label='RGB', marker='o', s=65)
ax.scatter(hybrid.hibrido_degradado, positions, label='Híbrido reportado', marker='s', s=65)
ax.set_yticks(positions, hybrid.modelo)
ax.invert_yaxis()
ax.set_xlim(0.5, 0.75)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'.replace('.', ',')))
ax.set_title('Frequência como complemento: três contrastes reportados')
ax.set_xlabel('ROC-AUC no teste degradado')
ax.legend(frameon=False)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)

## 3. As três perguntas de explicabilidade

**Tarefa 1:** congelar 64 imagens, com 16 VP, FN, VN e FP definidos pelo modelo de referência; reutilizar as mesmas identidades em todos os modelos. A distribuição 16×4 não é garantida para os demais modelos.

**Tarefa 2:** selecionar somente na validação um par competente RGB–frequência da mesma arquitetura, regime e semente. Comparar cada atribuição no domínio nativo: pixels em RGB, coeficientes no espectro. Não há heatmaps novos neste notebook.

**Tarefa 3:** calcular a interseção das falhas do conjunto declarado, sem descartar predições faltantes silenciosamente, e comparar com controles pareados por rótulo. Não chamar essas falhas de universais.

A semente principal é 42. Falsa é a classe positiva. O alvo das atribuições específicas de classe é a diferença entre o logit falso e o real.

## 4. Métodos e cuidados

**Gradientes e localização:** Grad-CAM, Integrated Gradients, Gradient SHAP, Saliency, Input × Gradient, SmoothGrad.

**Perturbação e aproximação local:** Kernel SHAP, LIME, Occlusion, Feature Ablation, Shapley Value Sampling.

**Atenção:** Attention Rollout.

O suporte depende da arquitetura. Attention Rollout é agnóstico à classe. Concordância visual não substitui checagens numéricas, sensibilidade ao baseline ou controles de randomização e perturbação.

## 5. Estado da evidência

- Nenhum novo experimento de explicabilidade foi executado para esta apresentação.
- ROC-AUC não é porcentagem de acerto; quedas e ganhos estão em unidades de AUC.
- Os gráficos exibem médias arredondadas, sem intervalos inferidos ou significância presumida.
- Falhas compartilhadas são relativas ao conjunto de checkpoints declarado, não universais.
- Atribuições não estabelecem causas nem permitem inferir atributos demográficos.

Nenhuma taxa de falhas compartilhadas ou conclusão demográfica foi estimada neste material. A execução futura fornece esses resultados; o código e a literatura não os predeterminam.

## Fontes

**[A] Cunha et al. — estudo anterior** — paper/original-sibgrapi.pdf, Tabela III e seção IV. [Consultar](https://github.com/Lucas-PG/FaceForgery-Benchmark/blob/e9beb4e212b7e3f749f65e84d27ef7f10b52de99/paper/original-sibgrapi.pdf)

**[B] Manuscrito principal — seis páginas** — paper/six-page/main.tex, seções III–VI. [Consultar](https://github.com/Lucas-PG/FaceForgery-Benchmark/blob/e9beb4e212b7e3f749f65e84d27ef7f10b52de99/paper/six-page/main.tex)

**[C] Guia de execução e configuração** — docs/explicability.md; configs/explicability.yaml. [Consultar](https://github.com/Lucas-PG/FaceForgery-Benchmark/blob/e9beb4e212b7e3f749f65e84d27ef7f10b52de99/docs/explicability.md)

**[D] Sundararajan et al. — Integrated Gradients** — ICML, 2017. [Consultar](https://proceedings.mlr.press/v70/sundararajan17a.html)

**[E] Selvaraju et al. — Grad-CAM** — ICCV, 2017. [Consultar](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html)

**[F] Adebayo et al. — Sanity Checks for Saliency Maps** — NeurIPS, 2018. [Consultar](https://arxiv.org/abs/1810.03292)

**[G] Hedström et al. — Quantus** — JMLR, 2023. [Consultar](https://jmlr.org/papers/v24/22-0142.html)

Referências completas da pesquisa: `paper/explicability/references.bib`.